In [3]:
!pip install -q fastapi uvicorn python-docx pypdf python-multipart python-dotenv openai pyngrok nest_asyncio

In [4]:
import io
import json
import os
import re
from dataclasses import dataclass, field
from datetime import date
from typing import Any, Dict, List, Optional, Union

from docx import Document
from pypdf import PdfReader
from dotenv import load_dotenv

try:
    from openai import OpenAI
except:
    OpenAI = None

load_dotenv()

False

In [5]:
ALLOWED_EXTENSIONS = {".pdf", ".docx", ".txt"}

GROQ_BASE_URL = "https://api.groq.com/openai/v1"
DEFAULT_GROQ_MODEL = "llama-3.1-8b-instant"
DEFAULT_OPENAI_MODEL = "gpt-4o-mini"

CURRENT_DATE = date.today()

MONTHS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}

In [7]:
@dataclass
class CandidateScore:
    filename: str
    technical_skills_match: int
    experience_match: int
    education_match: int
    overall_fit_score: int
    strengths: List[str]
    gaps: List[str]
    summary: str
    confidence_score: int = 0
    ats_breakdown: Dict[str, int] = field(default_factory=dict)
    evidence: List[str] = field(default_factory=list)
    red_flags: List[str] = field(default_factory=list)

    def as_dict(self):
        status = "Shortlisted" if self.overall_fit_score >= 50 else "Rejected"

        return {
            "filename": self.filename,
            "technical_skills_match": self.technical_skills_match,
            "experience_match": self.experience_match,
            "education_match": self.education_match,
            "overall_fit_score": self.overall_fit_score,
            "status": status,
            "strengths": self.strengths,
            "gaps": self.gaps,
            "summary": self.summary,
            "confidence_score": self.confidence_score,
            "ats_breakdown": self.ats_breakdown,
            "evidence": self.evidence,
            "red_flags": self.red_flags,
        }

In [8]:
def clean_text(text):
    return re.sub(r"\s+", " ", text or "").strip()

def cap_score(value):
    return max(0, min(100, round(value)))

In [9]:
def extract_text_from_file(filepath):
    ext = os.path.splitext(filepath)[1].lower()

    if ext == ".pdf":
        reader = PdfReader(filepath)
        return clean_text(
            " ".join(page.extract_text() or "" for page in reader.pages)
        )

    elif ext == ".docx":
        doc = Document(filepath)
        return clean_text(
            " ".join(p.text for p in doc.paragraphs)
        )

    elif ext == ".txt":
        with open(filepath, "r", encoding="utf-8") as f:
            return clean_text(f.read())

    else:
        raise ValueError("Unsupported file")

In [10]:
from google.colab import files

print("Upload Job Description File")
jd_upload = files.upload()

jd_file = list(jd_upload.keys())[0]
jd_text = extract_text_from_file(jd_file)

print("Upload Resume Files")
resume_uploads = files.upload()

resume_files = list(resume_uploads.keys())

print("JD:", jd_file)
print("Resumes:", resume_files)

Upload Job Description File


Saving Updated My_resume.pdf to Updated My_resume.pdf
Upload Resume Files


Saving My_resume _updated.pdf to My_resume _updated.pdf
JD: Updated My_resume.pdf
Resumes: ['My_resume _updated.pdf']


In [14]:
from openai import OpenAI

GROQ_API_KEY = "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"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [15]:
def score_resume(jd_text, resume_text):

    prompt = f"""
    Compare this resume against the JD.

    Return JSON with:
    {{
      "overall_fit_score": 0-100,
      "strengths": [],
      "gaps": [],
      "summary": ""
    }}

    JOB DESCRIPTION:
    {jd_text[:6000]}

    RESUME:
    {resume_text[:10000]}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{
            "role":"user",
            "content":prompt
        }],
        temperature=0.1
    )

    return response.choices[0].message.content

In [16]:
results = []

for resume_file in resume_files:

    resume_text = extract_text_from_file(resume_file)

    result = score_resume(
        jd_text,
        resume_text
    )

    results.append({
        "resume": resume_file,
        "result": result
    })

print("Done")

Done


In [17]:
for r in results:
    print("="*60)
    print(r["resume"])
    print(r["result"])

My_resume _updated.pdf
Based on the provided job description and resume, here's a comparison of the two:

**Overall Fit Score:** 85

**Strengths:**

1. **Strong technical skills**: The candidate has a good understanding of various programming languages, frameworks, and technologies, including Python, Java, JavaScript, SQL, NumPy, Pandas, Scikit-learn, Matplotlib, Seaborn, HTML, CSS, JavaScript, React, PHP, and SQL.
2. **Machine learning expertise**: The candidate has experience with machine learning, including building a model using Logistic Regression to predict job placement outcomes and implementing NLP and LLM-based techniques for question answering.
3. **Web development skills**: The candidate has built a dynamic multi-page e-commerce web application using React and TypeScript and developed an interactive web app using Streamlit.
4. **Competitive programming skills**: The candidate has solved 300+ diverse programming problems on LeetCode and GeeksforGeeks, demonstrating strong alg